# What the critic can and cannot learn

Every run in `runs/` reports a value function whose out-of-sample explained variance is
**negative**: `ABLATE+ent` −0.036, `ENT` −0.327, `NODETACH+ent` −0.103. A baseline worse
than the mean of the returns is not a critic, and this series has been calling itself
actor-critic PPO throughout.

`06` treated that as a defect to fix. This notebook is the record of trying — five
interventions, all of which made it worse or did nothing — and then of the measurement that
explains why, which is not about hyperparameters at all.

**The finding.** Prompts genuinely differ in difficulty: with `G=8` samples per prompt,
**67% of the reward variance is between prompts**. But the policy's own representation of
the prompt does not encode that difficulty. Held out *by prompt*, at the start of the
completion — where the state is essentially the prompt — the explained variance is
**0.000**, and a 256-unit MLP that reaches +0.965 in-sample generalises to **−0.030**.

The reward only becomes predictable once the answer has been written, rising to +0.21 by
the last fifth of the completion. By then the advantage is already determined.

So `V(s_t) ≈ constant` is very nearly the optimal value function here, and a critic that
converges to the batch mean — which is exactly what `|V − R| ≈ 0.0000x` says ours does — is
behaving correctly. Every intervention that pushed it to learn more had it fit noise.

**And the one thing that does work.** A critic reading *privileged* state — the case's true
`root_cause`, which the actor never sees — clears the bar immediately: median `value_ev`
**+0.42** against the baseline's −0.02, and it is the only configuration in the series that
ever exceeds +0.3 at all. That is not a trick; a baseline may be any function of the state,
and the label is determined by the prompt, so the policy gradient stays unbiased. It works
for exactly the reason the ordinary critic fails — it is handed the quantity that the
ordinary critic would have had to solve the task to compute.

**The consequence for this series.** GRPO's group-mean baseline is the constant-per-prompt
baseline, computed exactly from `G` samples rather than learned from a representation that
does not carry the signal. On this task the *learned, unprivileged* per-token critic's
ceiling is a quantity GRPO gets for free. The comparison table at the top of the README has
an answer, and for the unprivileged case it is negative.

## What was tried

Five in-loop interventions, all on `ABLATE + entropy_coef=0.005`, seed 0, against the same
baseline. `value_ev` is averaged over the run.

| run | change | `value_ev` | note |
| --- | --- | --- | --- |
| `ppo-qwen3-17b-ablate-ent-s0` | — (baseline) | −0.036 | |
| `ppo-qwen3-17b-ablate-ent-fix-s0` | `critic_window=25`, `recompute_advantages`, no value clip | −0.018 | 200 steps; held-out `cause_acc` **0.410 → 0.270** |
| `probe-A-cw25-ce50` | `critic_epochs` 8 → 50 | −0.234 | per-batch sd 0.134 → **1.077** |
| `probe-B-cw5-ce8` | `critic_window` 25 → 5 | −0.043 | |
| `probe-C-g4` | `samples_per_prompt` 1 → 4 | −2.171 | only 2 distinct prompts per batch |
| `probe-D-layer27` | `value_layer=-2`, rate unchanged | −2.6 | **my bug**, see below |
| `probe-E-layer27-lr` | `value_layer=-2`, `value_lr` rescaled | see below | |

Two of those deserve their own note because they were my errors, not the algorithm's.

**`probe-A` confounded two changes.** It raised `critic_epochs` six-fold *and* had value
clipping off. `value_clip_eps` exists to stop one batch dragging the baseline where the next
batch's advantages cannot recover from, and I removed that brake on the paper's advice (C13)
and then multiplied the steps. It is not a clean test of `critic_epochs`.

**`probe-D` was a learning-rate scaling bug.** `value_lr = 3e-5` was derived from the *last*
layer's feature norm, and `ppo_ac.py` documents the relation `|dV| ≈ lr · ‖h‖₁`. Layer 27's
`‖h‖₁` is **31042** against the last layer's **1605** — 19.3× — because the last layer has
passed the final norm. At the unchanged rate V moves 0.93 per step against a reward in
[0, 1]. `probe-E` rescales the rate to 1.55e-6, which reproduces the same `|dV|`.

In [ ]:
import sys, json, math, collections, statistics as st
from pathlib import Path
HERE = Path.cwd() if (Path.cwd() / "ppo_ac.py").exists() else Path("experiments/notebooks/smoke_test")
sys.path.insert(0, str(HERE.resolve()))
import ppo_ac


def critic_summary(run_dir, maxstep=80):
    rows = [json.loads(l) for l in open(HERE / "runs" / run_dir / "metrics.jsonl")]
    rows = [r for r in rows if r["step"] < maxstep]
    take = lambda k: [r[k] for r in rows if r.get(k) is not None]
    ev, fit = take("value_ev"), take("value_ev_fit")
    # The MEDIAN, not the mean. Explained variance divides by the batch's own
    # return variance, so a batch whose sequences all drew a similar reward
    # produces a huge negative outlier -- `undefined` only catches variance that
    # is exactly zero. Two such steps drag the privileged run's mean to -0.288
    # while its median is +0.258. Every run in `runs/` is reported by its mean.
    return {
        "median": st.median(ev), "mean": st.mean(ev),
        "over_bar": sum(1 for v in ev if v > 0.3) / len(ev),
        "value_ev_fit": st.median(fit),
        "value_std": st.mean(take("value_std")),
        "V_minus_R": abs(st.mean([r["value_mean"] - r["return_mean"] for r in rows])),
    }

RUNS = [
    ("baseline  layer28 lr3e-5",       "ppo-qwen3-17b-ablate-ent-s0"),
    ("F  PRIVILEGED (asymmetric)",     "probe-F-privileged"),
    ("window25 + recompute + noclip",  "ppo-qwen3-17b-ablate-ent-fix-s0"),
    ("A  critic_epochs 50",            "probe-A-cw25-ce50"),
    ("B  critic_window 5",             "probe-B-cw5-ce8"),
    ("C  samples_per_prompt 4",        "probe-C-g4"),
    ("D  layer27, rate unchanged",     "probe-D-layer27"),
    ("E  layer27, rate rescaled",      "probe-E-layer27-lr"),
]
print(f"{'run':34} {'MEDIAN':>8} {'mean':>8} {'>0.3':>7} {'v_std':>7} {'|V-R|':>8}")
for lab, d in RUNS:
    try:
        c = critic_summary(d)
        print(f"{lab:34} {c['median']:+8.3f} {c['mean']:+8.3f} {c['over_bar']:6.0%} "
              f"{c['value_std']:7.4f} {c['V_minus_R']:8.5f}")
    except FileNotFoundError:
        print(f"{lab:34}  (not run)")

# `|V - R|` is the tell that survives every configuration: the head's mean tracks the
# returns' mean to four or five decimals while explaining none of their variance. That is
# a constant, and on a target that is mostly noise a constant is the least-squares answer.

## The measurement that explains it

Three probes on the **frozen** policy, so nothing here is confounded by a moving target.
48 prompts × 8 samples = 384 rollouts, features from every hidden layer, held out **by
prompt** — never by token or by sequence, because with `gamma = lam = 1` and a terminal
reward every token of a sequence carries the same target, and every sequence of a prompt
shares its difficulty. Splitting either way leaks the answer.

### 1. Prompts do differ

| | |
| --- | --- |
| between-prompt variance | **66.9%** |
| within-prompt (sampling noise) | 33.1% |
| ceiling on `value_ev` for any critic at t=0 | **+0.669** |

So there is a great deal to learn, and the critic gets +0.031 of it.

### 2. But the prompt's representation does not carry it

Held out by prompt, target = the prompt's mean reward, first 15% of the completion:

| head on layer 26 | train EV | **test EV** |
| --- | --- | --- |
| instantaneous, linear | +0.370 | **−0.407** |
| instantaneous, MLP-64 | +0.196 | −0.145 |
| instantaneous, MLP-256 | +0.238 | −0.141 |
| running mean, linear | +0.650 | −0.188 |
| running mean, MLP-64 | +0.965 | −0.131 |
| running mean, MLP-256 | +0.931 | **−0.030** |

Every one is negative out of sample. A 256-unit MLP reaching +0.965 in-sample and −0.030
out is not a weak probe; it is a probe with nothing to find. This rules out "the linear head
was too small", which was the standing explanation in `ValueHead`'s docstring.

### 3. The signal arrives only as the answer is written

Explained variance by position, causal running mean, layer 26:

| position | test EV |
| --- | --- |
| first 15% | **−0.001** |
| 15–35% | +0.037 |
| 35–55% | +0.135 |
| 55–80% | +0.203 |
| 80–100% | **+0.208** |

The instantaneous feature never gets above +0.03 and is mostly negative; only pooling the
prefix helps, and only late.

### 4. Where the +0.993 in `ValueHead`'s docstring came from

Mean-pooling the *whole* completion — which is not causal, so no `V(s_t)` can use it —
layer 27 reaches +0.456. Fitting the same features with no held-out split at all reaches
+0.311, and splitting by token instead of by sequence reaches +0.101 against +0.031 for the
honest split. With 2049 parameters and only 200 independent targets, a probe can
interpolate; the docstring's number is that artefact, and the claim it supports — that the
representation is not the constraint — does not survive a split.

## The decisive measurement: difficulty *is* the answer

Everything above says the prompt's representation does not predict the reward. That leaves
two very different possibilities, and they lead opposite ways: either the signal is not in
the prompt at all (nothing can work), or it is there and the model cannot see it (an SFT
cold start might).

The task's own metadata settles it, with no model involved. `data/*.jsonl` carries the
variables the generator designed difficulty around — `tier` (easy/hard: whether the two feed
temperatures are equal, so whether `TCF` cancels and the arithmetic is a plain percent
change), `margin_pp` (how far the case sits from a flag threshold), and `severe`. Regressing
each against the prompt's mean reward over `G=8` samples, leave-one-out over 48 prompts:

| predictor of prompt-mean reward | LOO $R^2$ |
| --- | --- |
| `tier` + `margin_pp` + `severe` — the *designed* difficulty | **−0.081** |
| `root_cause` one-hot — the answer itself | **+0.902** |

The variables the task was built to make hard explain **nothing**. What explains 90% of it
is which root cause the case happens to have:

| root cause | prompt-mean reward |
| --- | --- |
| **biofouling** | **0.512** |
| **scaling** | **0.492** |
| oxidation_damage | 0.221 |
| mechanical_leak | 0.169 |
| colloidal_fouling | 0.167 |
| organic_fouling | 0.162 |
| compaction | 0.154 |

Those top two are exactly the labels the frozen policy emits — `06`'s gate cell measures the
frozen policy using 5 of 7 causes with 78% of its mass on the top two, and every trained run
narrowing further. A prompt is "easy" if its answer happens to be one of the labels the
policy already produces.

**So the value function's task is the policy's task.** To predict the reward from the
prompt, `V(s_0)` would have to determine the correct root cause and check it against what
the policy is about to emit. A critic that could do the first half would already be a solved
policy, and there would be nothing left to train.

This is why every intervention failed in the same way. The critic is not underfitting, badly
scaled, starved of data, or reading the wrong layer. It is being asked for something that
cannot be computed from its input without solving the problem the whole run exists to solve.
Converging immediately and precisely to the batch mean — `|V − R| ≈ 0.00001` — is the
correct answer to the question actually being posed.

In [ ]:
import numpy as np
# the numbers above, reproduced from the task metadata alone -- no model involved
cases = ppo_ac.load_cases("train")
rng = np.random.default_rng(0)
sel = [cases[i] for i in rng.choice(len(cases), 48, replace=False)]
print("designed difficulty variables:",
      sorted({k for c in sel for k in c["meta"]}))
print("tier values:", sorted({c["tier"] for c in sel}))

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(HERE / "runs" / "critic_verdict.png")))

## One critic that can work, and what it costs

The verdict above is about a critic reading the policy's hidden state. It is not a
statement that no baseline can beat a constant — only that none can be *computed from that
input* without solving the task.

There is a standard way out, and it is not a hack: **asymmetric actor–critic**, where the
critic reads privileged information the actor cannot see. Sim-to-real robotics uses it
routinely — the critic gets the simulator's true state, the actor gets pixels.

It is unbiased here, for the usual reason. The policy gradient is unchanged by any baseline
that is a function of the **state**:

$$\mathbb{E}\big[\nabla \log \pi(a \mid s)\, b(s)\big] = 0 \quad \text{for any } b.$$

The case's true `root_cause` is determined by the prompt, so it is part of $s$ and not of
$a$. Feeding it to the value head shifts no gradient in expectation; it only changes the
variance. The actor never sees it — it is concatenated to the value head's input, and
`value_detach=True` keeps the value loss off the trunk entirely.

And we already know it has the signal: `root_cause` alone explains **+0.902** of the
prompt-mean reward, leave-one-out, while the designed difficulty variables explain −0.081.

`privileged_cause=True` appends the one-hot to the cached features. It is off by default;
every run in `runs/` predates it.

**What it does not do.** It cannot make the policy better at the task — it only tells the
advantage which prompts were easy. And it is unavailable at inference, so an agent trained
this way has no privileged critic to fall back on; the critic is a training-time device
only. That is the normal situation for a value function, but worth stating plainly given
what it is being given.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(HERE / "runs" / "critic_interventions.png")))

## Does a working critic help the task?

The runs above measure the critic. They do not measure whether fixing it is worth anything,
and that is a separate question with a separate answer.

`ppo-qwen3-17b-ablate-ent-priv-s0` is the controlled version: 200 steps, `eval_every=25`,
same seed, same weights, same entropy coefficient, same `value_lr`, `critic_epochs` and
`value_clip_eps` as `ppo-qwen3-17b-ablate-ent-s0` — **one variable changed**.

| | baseline | privileged |
| --- | --- | --- |
| `value_ev` median | −0.009 | **+0.799** |
| steps with `value_ev > 0.3` | 0% | **88%** |
| `value_std` | 0.036 | **0.152** |
| **`adv_std`** | 0.0345 | **0.0216** |
| held-out `cause_acc` at step 200 | 0.410 | 0.425 |
| `flags_acc` | 0.588 | 0.513 |
| `numeric_acc` | 0.063 | 0.062 |

`adv_std` is the row that matters, and it is stronger evidence than `value_ev`. A baseline's
entire job is to reduce the variance of the gradient estimate; the advantage spread fell
**37%**. That is the mechanism working, measured directly, not a metric looking good.

And the task did not move. Across the eight evaluation points after step 0 the difference in
`cause_acc` averages **−0.0025**, ranging −0.045 to +0.050, against a seed spread of ±0.055
measured at this exact configuration (0.410 at seed 0, 0.300 at seed 1). The +0.015 at step
200 is inside that band, and so is every other point.

The chain is complete and every link is measured:

> critic fixed → gradient variance −37% → task unchanged

So **gradient variance is not the binding constraint on this task.** That is worth more than
the critic fix itself, because it rules out a whole family of interventions at once: larger
batches, more rollouts per step, better advantage estimators, tighter trust regions — every
one of them buys variance, and variance has now been shown to be free.

What remains binding is what `06`'s gate cell measures: the label collapse. A baseline
reduces the variance of the gradient; it does not change its expectation, and it does not
choose which two or three of the seven causes the policy falls into.

### Is 0.425 the best result in the series?

It is the highest number, and it is not a better policy. Running the same greedy evaluation
`06`'s gate cell reads:

| run | `cause_acc` | ceiling | labels used | H | at % of ceiling | `exact_match` |
| --- | --- | --- | --- | --- | --- | --- |
| frozen | 0.235 | 0.435 | 5/7 | 0.61 | 54% | 0.000 |
| `PROBE` | 0.335 | 0.615 | 5/7 | 0.69 | 54% | 0.000 |
| `ABLATE+ent` | 0.410 | 0.440 | 4/7 | 0.57 | 93% | 0.000 |
| **`ABLATE+ent+priv`** | **0.425** | 0.435 | **4/7** | 0.57 | **98%** | **0.000** |

Three things follow, and none of them is "best result":

* **+0.015 over the previous high is inside the seed band** (±0.055 at this configuration).
* **It is the most saturated run in the series** — 98% of a ceiling set by emitting three
  causes (biofouling 84, organic_fouling 59, scaling 56 of 200). It is not discriminating
  better; it is guessing from a shorter list that happens to be better aligned.
* **It fails `06`'s gate** on `entropy_not_below_frozen` (0.57 against the frozen policy's
  0.61), exactly like every other trained run.

And `exact_match` is **0.000**, as it is at all 27 evaluation points of every run in `runs/`.
Across the whole series the model has never once produced a completely correct answer.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(HERE / "runs" / "privileged_critic.png")))

## The verdict

A critic that reads the policy's hidden state is not fixable on this task, and the reason is
a property of the task rather than of the implementation:

* The reward is essentially "did the model name the right root cause". That is decided by
  the completion, not by the prompt.
* The policy's representation of the prompt does not predict which prompts it will get
  right — measured at +0.000 held out, with linear and nonlinear heads.
* By the time the state does predict the reward, the completion is written and the
  advantage is already determined.

Therefore the best causal value function available is very nearly a constant, and the
constant is the batch mean. That is what our critic learns, immediately and precisely, in
every configuration tried.

**What this says about the comparison the series exists to make.** `README.md` opens with a
table contrasting a learned `V(s_t)` against GRPO's group mean. The learned baseline's
ceiling here is the per-prompt constant, and GRPO computes exactly that from its `G` samples
without needing the representation to carry anything. The per-token credit assignment that
actor-critic PPO offers has nothing to assign: with `gamma = lam = 1` and a terminal reward,
every token of a sequence has the same return.

**What would change the answer**, and none of it is a hyperparameter:

* A **dense or intermediate reward** — something scored before the answer is complete —
  would give `V(s_t)` a reason to vary along the sequence.
* A **prompt-difficulty signal the model can read**. The task's own `tier` field (easy/hard,
  determined by whether the two feed temperatures are equal) is exactly such a signal and is
  *in* the prompt; that the policy's hidden state does not expose it is a statement about
  the frozen model, and an SFT cold start might change it.
* `gamma < 1`, which would make the return vary within a sequence and give the critic a
  reason to exist. It also introduces bias, and there is no reason to want it here.

* **Privileged state**, which is what `privileged_cause` does and what the runs above
  measure. It is the only one of these that has been tried, and it works.

Until one of the first three changes, the honest configuration for an *unprivileged* critic
is `value_detach=True` with the critic understood as a slow estimator of a constant — or
GRPO, which computes that constant exactly.

### A note on `value_ev` itself

The mean of `value_ev` over a run is not a usable summary, and this notebook nearly drew the
wrong conclusion from it. Explained variance divides by the batch's own return variance, so
a batch where every sequence drew a similar reward produces a large negative outlier;
`undefined` only catches variance that is exactly zero. The privileged run's mean is -0.314
and its median is **+0.419**, because 10 of its 80 steps scored below −0.5 (worst -19.9).
Every run in `runs/` is reported by its mean. The median, and the fraction of steps above a
threshold, are the honest statistics — that is what the figure above plots.

---

## What the critic was being asked to predict

Everything above measures what a value function *can* learn from the policy's state. It does
not ask what there was to learn, and that turns out to be the shorter story — and the one
that does not need a probe.

### Two components carry no variance, and the largest one is noise

`ABLATE` is the weight set every run from `04` on trains against. Measured against what each
weight allows, on the `ablate_ent` greedy evaluation:

| component | weight | earned | of the weight | |
| --- | --- | --- | --- | --- |
| `format` | 0.10 | 0.1000 | **100%** | saturated on every case — a constant |
| `stage` | 0.03 | 0.0300 | **100%** | likewise |
| `numeric` | **0.35** | 0.0095 | **2.7%** | |
| `flags` | 0.12 | 0.0706 | 59% | downstream of the same three numbers |
| `root_cause` | 0.25 | 0.1845 | 74% | |
| `action` | 0.15 | 0.0457 | 31% | downstream again |

`format` and `stage` are earned in full on every case, so they contribute nothing to the
*variance* of the reward: a critic that predicts them perfectly explains none of it.
`numeric` carries the second-largest weight in the set and returns 2.7% of it. What is left
holding both variance and signal is `root_cause` — and `root_cause` is exactly the term the
sections above showed a critic cannot predict without solving the task.

**So no reward component both varies and is predictable from the prompt.** The constant part
is constant, and the variable part is either the answer itself or noise. `value_ev ≈ 0`
follows from how the reward is built, before any argument about layers, windows or head
capacity — and before the probes in §3 that took a card to run.

### The arithmetic is not the temperature correction. It is the arithmetic

`06`'s appendix concluded the model is not doing the computation at all, and argued it from
`cause_acc` split by tier (0.403 `easy` against 0.423 `hard`). That is the wrong metric for
the claim. `numeric` is stored per case and splits the same way, much more sharply — the
fraction of the three numbers landing within the 0.5 pp tolerance:

| run | `easy` (TCF cancels) | `hard` |
| --- | --- | --- |
| frozen | 0.036 | 0.033 |
| `MAIN` | 0.023 | 0.014 |
| `PROBE+ent` | 0.116 | 0.094 |
| `ABLATE+ent` | 0.067 | 0.056 |
| `ABLATE+ent+priv` | 0.072 | 0.042 |

Now look at what the three numbers actually require. Only `normalized_flow_change_pct` uses
`TCF(T) = 1.03 ** (25 - T)`. `salt_passage_change_pct` is a division and a percent change;
`dp_change_pct` is a sum and a percent change. **Two of the three numbers involve no domain
formula in any case, easy or hard**, and they are missed at the same rate as the third. Over
the 1000 case-evaluations in `runs/paired/`, the count of numbers within tolerance is 844
zeros, 140 ones, 15 twos and **one three**.

What is absent is not the RO correction factor, and it is not knowledge: the task is
closed-book by construction, with the formula, the thresholds, the rules and the schema all
supplied in the prompt. What is absent is multi-step decimal arithmetic carried to one
decimal place. That is why `exact_match` is 0.000 at all 27 evaluation points of every run
in `runs/` — `exact_match` needs the numbers.

### The diagnosis half is not capability-limited

The same per-case records, joined against the dev labels, separate the two failure modes
cleanly. Recall on the causes a run actually emits, against the causes it never emits:

| run | causes emitted | recall on those | on the rest | `cause_acc` |
| --- | --- | --- | --- | --- |
| frozen | 4/7 | 47/114 = 0.412 | 0/86 | 0.235 |
| `PROBE+ent` | 4/7 | 73/114 = 0.640 | 0/86 | 0.365 |
| `ABLATE+ent` | 4/7 | 82/115 = 0.713 | 0/85 | 0.410 |
| **`ABLATE+ent+priv`** | **3/7** | **85/87 = 0.977** | 0/113 | 0.425 |

The privileged run is right on **85 of the 87** cases whose true cause is one of the three
labels it will say, and on 0 of the other 113. Its 0.425 is 87/200 × 0.977. That is not a
model that cannot tell the causes apart; it is a model that will only say three of the
words. (The comparison table above counts it as 4/7 because `predicted_cause_hist` includes
a single `null`; three is the count of real causes.)

And every cause is reachable — just not in the same run. `PROBE+ent` covers
`oxidation_damage`, `organic_fouling` and `compaction`, which are exactly the ones the other
runs drop, at 64% recall. Six of the seven causes are learned by *some* run in this series;
only `mechanical_leak` is 0 everywhere.

So **coverage, not discrimination, is what caps `cause_acc`** — and a baseline cannot touch
coverage. A baseline rescales the gradient of actions that appear in a sample; a label that
is never sampled has no gradient to rescale. That is the mechanical reason
`critic fixed → gradient variance −37% → task unchanged` had to come out the way it did.

### What each failure mode costs

* **The arithmetic is absent from the model**, and it is absent at the level of elementary
  decimal arithmetic rather than the domain formula. No reward shapes it into existence and
  no baseline reduces its variance. The options are a calculator tool or a larger model.
* **The label collapse is a training dynamic**, and it is fixable at this model size. The
  frozen policy emits 5 causes at entropy 0.61 and every trained run is narrower — a KL
  penalty to the frozen policy, a coverage term in the reward, or an SFT cold start on a
  balanced label prior all act on it directly.

And one thing **not** to do, because the series has already run the experiment. `ABLATE`
spends 0.35 of its reward on a component that returns 2.7% of it, so moving that weight onto
`root_cause` looks free. It is not: coverage tracks the `root_cause` weight inversely, across
all three weight sets.

| weight set | `root_cause` weight | causes emitted | `cause_acc` |
| --- | --- | --- | --- |
| `MAIN` | **0.45** | **3/7** | 0.170 — *below* the frozen policy's 0.235 |
| `ABLATE` | 0.25 | 4/7 | 0.245 |
| `PROBE` | **0.10** | **5/7** | 0.335 |

The heavier the weight on the label, the harder the policy collapses onto its best few — the
in-set recall barely moves (0.405 / 0.430 / 0.469) while the number of causes it will say
falls. `MAIN` being the worst configuration in the series is this, measured. Redistributing
the dead `numeric` weight onto `root_cause` would buy more collapse; the honest destinations
are `flags`, or a coverage term that does not exist yet.

In [ ]:
# The two tables below, from the greedy evaluations in `runs/paired/` and the
# task's own labels. No model and no torch: both are re-reads of stored results.
DEV = Path("/home/bayan/MembraneClaw/experiments/membrane_grpo/data/dev.jsonl")
TRUTH = {json.loads(l)["id"]: json.loads(l)["answer"]["root_cause"] for l in open(DEV)}
ABLATE_W = dict(  # membrane_grpo/reward.py, the weight set every run from 04 on uses
    format=0.10, numeric=0.35, flags=0.12, stage=0.03, root_cause=0.25, action=0.15)

def paired(name):
    return json.load(open(HERE / "runs" / "paired" / f"{name}.json"))["overall"]

print("reward components, `ablate_ent` -- earned against what the weight allows")
ov = paired("ablate_ent")
for k, w in ABLATE_W.items():
    got = ov["components"][k]
    print(f"  {k:12} weight {w:.2f}   earned {got:.4f}   {got/w:6.1%} of it")

print("\nnumeric accuracy by tier -- `easy` is the tier where TCF cancels")
print(f"  {'run':22}{'easy':>8}{'hard':>8}   (fraction of the 3 numbers within 0.5 pp)")
for name in ("base", "main", "probe_ent", "ablate_ent", "ablate_ent_priv"):
    pc = paired(name)["per_case"]
    cols = [f"{sum(c['numeric'] for c in pc if c['tier'] == t) / (3 * sum(c['tier'] == t for c in pc)):>8.3f}"
            for t in ("easy", "hard")]
    print(f"  {name:22}" + "".join(cols))
allthree = collections.Counter(
    c["numeric"] for n in ("base", "main", "probe_ent", "ablate_ent", "ablate_ent_priv")
    for c in paired(n)["per_case"])
print(f"  numbers right per case, over all 1000 case-evaluations: {dict(sorted(allthree.items()))}")

print("\nroot_cause: recall on the labels a run emits, against the labels it never emits")
for name in ("base", "probe_ent", "ablate_ent", "ablate_ent_priv"):
    ov = paired(name)
    emitted = {k for k in ov["predicted_cause_hist"] if k in set(TRUTH.values())}
    hit = sum(c["cause"] for c in ov["per_case"])
    inside = [c for c in ov["per_case"] if TRUTH[c["id"]] in emitted]
    print(f"  {name:18} {len(emitted)}/7 labels   "
          f"in-set {hit}/{len(inside)} = {hit/len(inside):.3f}   "
          f"out-of-set 0/{200 - len(inside)}   cause_acc {ov['cause_acc']:.3f}")

print("\ncoverage against the `root_cause` weight -- the three weight sets, no entropy bonus")
RC_W = {"main": 0.45, "ablate": 0.25, "probe": 0.10}
for name, w in RC_W.items():
    ov = paired(name)
    emitted = {k for k in ov["predicted_cause_hist"] if k in set(TRUTH.values())}
    inside = [c for c in ov["per_case"] if TRUTH[c["id"]] in emitted]
    hit = sum(c["cause"] for c in ov["per_case"])
    print(f"  {name:8} root_cause weight {w:.2f}   {len(emitted)}/7 causes   "
          f"in-set recall {hit/len(inside):.3f}   cause_acc {ov['cause_acc']:.3f}")